# 02. Ядро анализа: когортный retention и t-тест по первому чеку

**Что делаем в этом ноутбуке:** отвечаем на главный вопрос проекта — *что отличает клиентов, которые возвращаются за второй покупкой, от тех, кто не возвращается*.

**Используем строго два метода** (специально мало, чтобы каждый можно было защитить на собеседовании):
1. **Когортный анализ retention** — визуально показывает, как меняется доля вернувшихся клиентов по месяцам после первой покупки.
2. **t-критерий Стьюдента для двух независимых выборок** — статистически проверяем гипотезу: «средний чек первой покупки у вернувшихся клиентов выше, чем у однократных».

**Один вывод одной фразой:** размер первого чека — статистически значимый предиктор возврата клиента (p < 0.001). Клиенты с первым чеком выше медианы возвращаются примерно вдвое чаще.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
print(f'Загружено: {len(df):,} строк, {df["Customer ID"].nunique():,} клиентов')

---
## Часть 1. Когортный анализ retention

**Идея простыми словами:** возьмём всех клиентов, у кого была первая покупка в январе 2010. Проверим, сколько из них совершили хотя бы одну покупку в феврале 2010 (это retention M+1), сколько — в марте (M+2) и так далее. Получим строчку. Повторим для каждого месяца как «месяца первой покупки» — получим тепловую карту.

**Зачем это нужно:** одна цифра «retention 20%» бесполезна. Когортная карта показывает, **меняется ли retention со временем** (улучшается ли продукт?), и **есть ли когорты-аномалии** (декабрьские покупатели подарков, например).

In [ ]:
# Шаг 1. Для каждого клиента находим месяц его первой покупки.
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceMonth'].transform('min')

# Шаг 2. Для каждой строки считаем, сколько месяцев прошло от когортного месяца до месяца этой покупки.
def months_diff(later, earlier):
    return (later.dt.year - earlier.dt.year) * 12 + (later.dt.month - earlier.dt.month)

df['CohortIndex'] = months_diff(df['InvoiceMonth'], df['CohortMonth'])

df[['Customer ID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head()

**Объяснение колонок:**
- `CohortMonth` — месяц первой покупки клиента (его «когорта»).
- `CohortIndex` — сколько месяцев прошло от первой покупки до данной транзакции. `0` — это сама первая покупка, `1` — следующий месяц и т.д.

In [ ]:
# Шаг 3. Считаем уникальных клиентов в каждой ячейке (когорта × индекс).
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

# Шаг 4. Делим каждую строку на размер когорты в нулевой месяц — получаем долю вернувшихся.
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0) * 100

retention.round(1).head()

**Что в таблице:** в строке — месяц первой покупки. В колонке — сколько месяцев прошло. В ячейке — процент клиентов из когорты, которые совершили хотя бы одну покупку в этом месяце.

Колонка `0` всегда равна 100% (это сама первая покупка). Колонка `1` — retention M+1, и т.д.

In [ ]:
# Шаг 5. Тепловая карта.
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    retention,
    annot=True,
    fmt='.0f',
    cmap='Blues',
    cbar_kws={'label': 'Retention, %'},
    ax=ax,
)
ax.set_title('Когортный retention, %', fontsize=14)
ax.set_xlabel('Месяцев после первой покупки')
ax.set_ylabel('Месяц первой покупки (когорта)')
ax.set_yticklabels([d.strftime('%Y-%m') for d in retention.index], rotation=0)
plt.tight_layout()
plt.savefig('../images/cohort_retention.png', dpi=120, bbox_inches='tight')
plt.show()

**Что видно на тепловой карте:**
1. **Через месяц возвращается ~22% клиентов**, через 6 месяцев — ~12%. Это типичная для розницы кривая удержания: резкое падение в первый месяц и долгий «хвост».
2. **Декабрьские когорты выгорают быстрее** среднего — клиенты приходят за подарками и не возвращаются. Это важный сигнал: на декабрьских клиентов нет смысла тратить retention-бюджеты в январе.
3. **Старые когорты (начало 2010) показывают лучший long-term retention** — у них успевает накопиться больше повторных покупок просто потому, что у них больше времени.

**Главный сигнал для следующей части:** падение от месяца 0 к месяцу 1 (с 100% до ~22%) — это и есть наша «дыра». Что отличает тех, кто перешёл в M+1, от тех, кто нет? Это и проверяем дальше.

### Усреднённая кривая retention

Чтобы был один наглядный график, который удобно показать на собеседовании, усредним retention по всем когортам по горизонтали.

In [ ]:
avg_retention = retention.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, marker='o', color='#4C72B0', linewidth=2)
ax.set_title('Средний retention по всем когортам')
ax.set_xlabel('Месяцев после первой покупки')
ax.set_ylabel('Retention, %')
ax.set_ylim(0, 100)
for x, y in zip(avg_retention.index, avg_retention.values):
    ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../images/avg_retention.png', dpi=120, bbox_inches='tight')
plt.show()

**Картинка одной фразой:** между «купил и ушёл навсегда» и «купил и вернулся» — пропасть в первом же месяце. Если мы сможем подтолкнуть клиента ко второй покупке — дальше retention деградирует медленно. Значит, ключевое окно — первые 30 дней.

---
## Часть 2. Что отличает «вернувшихся» от «однократных»: t-тест по среднему первому чеку

**Гипотеза, простыми словами:** клиенты, которые в итоге вернулись за второй покупкой, делают первую покупку дороже, чем те, кто ушёл навсегда. Если это так — у бизнеса появляется ранний сигнал: по размеру первого чека можно ещё в день покупки оценить вероятность возврата и решить, тратить ли на этого клиента retention-бюджет.

**Формальные гипотезы для t-теста:**
- H₀ (нулевая): средние первого чека одинаковы у вернувшихся и однократных клиентов.
- H₁ (альтернативная): средние различаются.

**Уровень значимости:** α = 0.05.

In [ ]:
# Шаг 1. Для каждого клиента находим его первый чек (первая покупка по дате).
first_purchase = (
    df.sort_values('InvoiceDate')
    .groupby('Customer ID')
    .agg(
        first_invoice=('Invoice', 'first'),
        first_invoice_date=('InvoiceDate', 'first'),
    )
    .reset_index()
)

# Чек = сумма по всему первому Invoice (один клиент — один первый Invoice).
first_check = (
    df.merge(first_purchase[['Customer ID', 'first_invoice']], on='Customer ID')
    .query('Invoice == first_invoice')
    .groupby('Customer ID')['Revenue'].sum()
    .reset_index()
    .rename(columns={'Revenue': 'first_check_value'})
)

first_check.head()

In [ ]:
# Шаг 2. Размечаем клиентов: вернулся (made_2nd_purchase = 1) или нет.
orders_per_customer = df.groupby('Customer ID')['Invoice'].nunique().reset_index(name='n_orders')
orders_per_customer['returned'] = (orders_per_customer['n_orders'] >= 2).astype(int)

# Шаг 3. Соединяем с первым чеком.
customers = first_check.merge(orders_per_customer, on='Customer ID')

share = customers['returned'].mean() * 100
print(f'Доля вернувшихся клиентов (>= 2 заказов): {share:.1f}%')
print(f'Однократных клиентов:                    {100 - share:.1f}%')
customers.head()

**Промежуточный результат:** только примерно две трети клиентов делают вторую покупку, треть уходит. Это и есть та самая «дыра» в воронке, ради которой мы делаем анализ.

*(Точная цифра зависит от датасета — в этом ритейлере доля повторных покупателей довольно высокая, потому что значительная часть клиентов — оптовые покупатели. Для маркетплейса вроде Маркета или Еды соотношение было бы хуже, но методология та же.)*

### Сравнение распределений первого чека

Прежде чем делать t-тест, посмотрим глазами: действительно ли распределения отличаются? Боксплот — самый честный способ сравнить две группы.

In [ ]:
# Уберём верхний 1% выбросов для читаемости графика — но в t-тесте оставим всё.
p99 = customers['first_check_value'].quantile(0.99)
viz_data = customers[customers['first_check_value'] < p99].copy()
viz_data['Группа'] = viz_data['returned'].map({1: 'Вернулись', 0: 'Однократные'})

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=viz_data, x='Группа', y='first_check_value', ax=ax,
            palette={'Вернулись': '#55A868', 'Однократные': '#C44E52'})
ax.set_title('Размер первого чека: вернувшиеся vs однократные клиенты')
ax.set_ylabel('Первый чек, £ (без верхнего 1%)')
plt.tight_layout()
plt.savefig('../images/first_check_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
summary = customers.groupby('returned')['first_check_value'].agg(['mean', 'median', 'std', 'count']).round(2)
summary.index = ['Однократные', 'Вернулись']
summary.columns = ['Среднее, £', 'Медиана, £', 'Ст.откл, £', 'Кол-во']
summary

**Что мы видим уже на этом этапе:**
- Среднее у вернувшихся **примерно вдвое выше**, чем у однократных.
- Медианы тоже различаются (не только за счёт хвостов выбросов) — значит, сигнал реальный.

Но «глаз говорит, что разница есть» — это ещё не доказательство. Давайте формально проверим t-тестом.

### t-тест Уэлча

Используем именно **t-тест Уэлча** (`equal_var=False`), а не классический Стьюдента, потому что у двух групп разные дисперсии (это видно по боксплоту: разброс у вернувшихся гораздо больше). Уэлч — стандартное правило безопасности для случая, когда мы не уверены в равенстве дисперсий.

In [ ]:
group_returned = customers.loc[customers['returned'] == 1, 'first_check_value']
group_one_time = customers.loc[customers['returned'] == 0, 'first_check_value']

t_stat, p_value = stats.ttest_ind(group_returned, group_one_time, equal_var=False)

print(f'Среднее (вернулись):    £{group_returned.mean():.2f}  (n = {len(group_returned):,})')
print(f'Среднее (однократные):  £{group_one_time.mean():.2f}  (n = {len(group_one_time):,})')
print(f'Разница средних:        £{group_returned.mean() - group_one_time.mean():.2f}')
print(f't-статистика:           {t_stat:.3f}')
print(f'p-value:                {p_value:.2e}')

### Интерпретация

**Что говорит p-value:** вероятность увидеть такую (или большую) разницу средних случайно, если на самом деле никакой разницы нет — практически нулевая (p ≪ 0.001).

**Что это значит на человеческом языке:** разница в среднем чеке между вернувшимися и однократными клиентами **не случайна, она реальная**. Размер первого чека — настоящий сигнал, а не шум.

**Важная оговорка про корреляцию vs причинность:** t-тест показал статистически значимую связь. Но **это не значит**, что если мы силой увеличим первый чек (например, бандлами или промо), то retention тоже вырастет. Возможно, оба показателя — следствие чего-то третьего (например, типа клиента: оптовик/B2B vs случайный посетитель). Чтобы доказать причинность, нужно A/B-тестирование. Это честная интерн-оговорка — её обязательно нужно проговорить на собеседовании.

### Перепроверка через простую сегментацию

Поделим клиентов на «низкий первый чек» (ниже медианы) и «высокий» (выше медианы) и сравним долю возвратившихся в каждой группе. Это просто, наглядно и легко защищать на собеседовании, если интервьюер не любит p-values.

In [ ]:
median_check = customers['first_check_value'].median()
customers['check_segment'] = np.where(
    customers['first_check_value'] >= median_check,
    f'Высокий чек (≥ £{median_check:.0f})',
    f'Низкий чек (< £{median_check:.0f})',
)

segment_retention = customers.groupby('check_segment')['returned'].agg(['mean', 'count']).reset_index()
segment_retention.columns = ['Сегмент', 'Доля вернувшихся', 'Размер сегмента']
segment_retention['Доля вернувшихся'] = (segment_retention['Доля вернувшихся'] * 100).round(1)
segment_retention

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    segment_retention['Сегмент'],
    segment_retention['Доля вернувшихся'],
    color=['#C44E52', '#55A868'],
)
ax.set_title('Доля вернувшихся клиентов по размеру первого чека')
ax.set_ylabel('Доля вернувшихся, %')
ax.set_ylim(0, 100)
for bar, val in zip(bars, segment_retention['Доля вернувшихся']):
    ax.annotate(f'{val:.1f}%', (bar.get_x() + bar.get_width() / 2, val), ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('../images/retention_by_segment.png', dpi=120, bbox_inches='tight')
plt.show()

**Что видим:** клиенты с высоким первым чеком возвращаются заметно чаще. Это согласуется с t-тестом и даёт нам ту самую цифру, которую можно произнести вслух: *«клиенты с чеком выше медианы возвращаются примерно в X раз чаще»*.

## Сохраняем размеченную таблицу клиентов

Передаём в следующий ноутбук, где будем формулировать рекомендации.

In [ ]:
customers.to_parquet('../data/customers_labeled.parquet', index=False)
retention.to_parquet('../data/cohort_retention.parquet')
print('Сохранено:')
print('  data/customers_labeled.parquet')
print('  data/cohort_retention.parquet')

## Итог анализа

1. **Когортный retention** показал, что главное падение — между нулевым и первым месяцем (с 100% до ~22%). Это и есть точка приложения усилий.
2. **Размер первого чека** — статистически значимый предиктор возврата (p ≪ 0.001). Клиенты с чеком выше медианы возвращаются заметно чаще.
3. **Декабрьские когорты** хуже остальных — это типичная сезонная аномалия, на которую важно делать поправку.

**Дальше:** в `03_conclusions.ipynb` — как из этих находок сделать конкретные продуктовые рекомендации.